# XGBoost vs Linear Regression - Data Nasabah
Ganti `DATA_PATH` dan `TARGET` sesuai datamu.

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

DATA_PATH = "data_nasabah.csv"   # <- ganti
TARGET = "target"                # <- ganti
df = pd.read_csv(DATA_PATH)
print(df.shape); df.head()

In [ ]:
# Kecilkan memori & siapkan fitur
for c in df.select_dtypes("float64"): df[c] = df[c].astype("float32")
for c in df.select_dtypes("object"): df[c] = df[c].astype("category")
X = df.drop(columns=[TARGET]); y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
# Linear Regression (kategori di-one-hot, NaN diisi median)
Xl_train = pd.get_dummies(X_train); Xl_test = pd.get_dummies(X_test).reindex(columns=Xl_train.columns, fill_value=0)
med = Xl_train.median(); Xl_train = Xl_train.fillna(med); Xl_test = Xl_test.fillna(med)
lr = LinearRegression().fit(Xl_train, y_train)
pred_lr = lr.predict(Xl_test)

In [ ]:
# XGBoost (CPU, hist, semua core)
xgb = XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                   tree_method="hist", enable_categorical=True, n_jobs=-1, early_stopping_rounds=50, random_state=42)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)
pred_xgb = xgb.predict(X_test)

In [ ]:
def ev(name, p): return {"model":name,"RMSE":mean_squared_error(y_test,p)**0.5,"MAE":mean_absolute_error(y_test,p),"R2":r2_score(y_test,p)}
pd.DataFrame([ev("LinearRegression",pred_lr), ev("XGBoost",pred_xgb)])

In [ ]:
import matplotlib.pyplot as plt
pd.Series(xgb.feature_importances_, index=X_train.columns).nlargest(15).sort_values().plot.barh(title="Top 15 Feature Importance (XGBoost)")
plt.tight_layout(); plt.show()